# QC Cross-Attention Source Importance
Ablation-based pie charts (X-cell style) + attention weight analysis.

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── Config ───────────────────────────────────────────────────────────────────
BASE     = Path("/dcai/users/hilarn/55_cu_0055/code/enhance_state/results/31")
RUN_ID   = "31"
CKPT_TAG = "eval_best.ckpt"

DATASETS = {
    "Replogle": BASE / f"qc_emb_lr1e-4/qc_emb_{RUN_ID}_lr1e-4",
    "Tian"    : BASE / f"qc_emb_Tian_lr1e-5/qc_emb_Tian_{RUN_ID}_lr1e-5",
}

METRIC = "pearson_delta"

SOURCE_NAMES = [
    "GTEx", "GWASAtlas", "ESM-2", "DepMap", "CellPainting",
    "GeneGenePT", "consensus", "neuronal_PPI", "pathway_consensus",
]
N_SOURCES = len(SOURCE_NAMES)

def read_agg_metric(eval_dir: Path, metric: str = METRIC):
    files = list(eval_dir.glob("*_agg_results.csv"))
    if not files:
        return None
    vals = []
    for f in files:
        df = pd.read_csv(f)
        if "statistic" in df.columns:
            df = df[df["statistic"] == "mean"]
        if metric in df.columns:
            vals.append(float(df[metric].values[0]))
    return float(np.mean(vals)) if vals else None

print("Source order:")
for i, s in enumerate(SOURCE_NAMES):
    print(f"  {i}: {s}")

In [ ]:
# ── Load ablation results and build importance scores ─────────────────────────
results = {}

for dataset, run_dir in DATASETS.items():
    full_score = read_agg_metric(run_dir / CKPT_TAG)
    ablated_scores = []
    for s in range(N_SOURCES):
        abl_dir = run_dir / f"ablate_source_{s}" / CKPT_TAG
        score = read_agg_metric(abl_dir)
        ablated_scores.append(score)
    results[dataset] = {"full": full_score, "ablated": ablated_scores}
    print(f"\n{dataset}  full={full_score:.4f}")
    for s, sc in enumerate(ablated_scores):
        score_str = f"{sc:.4f}" if sc is not None else "N/A"
        drop = (full_score - sc) if (sc is not None and full_score is not None) else float("nan")
        print(f"  ablate {SOURCE_NAMES[s]:15s}  score={score_str}  drop={drop:+.4f}")

In [ ]:
PALETTE = [
    "#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f",
    "#edc948", "#b07aa1", "#ff9da7", "#9c755f",
]

for dataset, res in results.items():
    full  = res["full"]
    drops = np.array([
        max(0.0, full - sc) if sc is not None else 0.0
        for sc in res["ablated"]
    ])

    total = drops.sum()
    if total == 0:
        print(f"{dataset}: all drops are zero — no data?")
        continue

    order  = np.argsort(drops)[::-1]
    labels = [SOURCE_NAMES[i] for i in order]
    sizes  = drops[order] / total
    colors = [PALETTE[i % len(PALETTE)] for i in order]

    fig, ax = plt.subplots(figsize=(7, 7))

    wedges, texts, autotexts = ax.pie(
        sizes, labels=None, colors=colors,
        autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
        startangle=90,
        wedgeprops=dict(linewidth=0.8, edgecolor="white"),
        pctdistance=0.78,
    )
    for at in autotexts:
        at.set_fontsize(9)

    ax.set_title(f"{dataset}\n(full {METRIC}={full:.3f})", fontsize=13, pad=16)
    legend_labels = [f"{lbl}  ({d:.3f} drop)" for lbl, d in zip(labels, drops[order])]
    ax.legend(wedges, legend_labels, loc="lower center",
              bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=9, frameon=False)

    plt.suptitle("Prior Knowledge Database importance", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(BASE / f"ablation_pie_{dataset}.pdf", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Ablation drop per perturbation: full vs ablate_source_7 (neuronal_PPI) ───
NPPI_SOURCE_IDX = 7

def load_pert_results(eval_dir: Path):
    files = [f for f in eval_dir.glob("*_results.csv") if "_agg_" not in f.name]
    if not files:
        return None
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        df["cell_type"] = f.name.replace("_results.csv", "")
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

tian_run = DATASETS["Tian"]
full_df  = load_pert_results(tian_run / CKPT_TAG)
abl_df   = load_pert_results(tian_run / f"ablate_source_{NPPI_SOURCE_IDX}" / CKPT_TAG)

if full_df is None or abl_df is None:
    print("Missing results — run predict first.")
else:
    # Detect pert column
    pert_col = next((c for c in ["perturbation", "pert", "gene", "condition"]
                     if c in full_df.columns), None)

    merged = full_df[[pert_col, "cell_type", "pearson_delta"]].merge(
        abl_df[[pert_col, "cell_type", "pearson_delta"]],
        on=[pert_col, "cell_type"], suffixes=("_full", "_ablated")
    )
    merged["drop"]     = merged["pearson_delta_full"] - merged["pearson_delta_ablated"]
    merged["has_nppi"] = merged[pert_col].isin(covered_symbols)

    # Display table sorted by drop
    display_cols = [pert_col, "cell_type", "pearson_delta_full", "pearson_delta_ablated", "drop", "has_nppi"]
    print(merged[display_cols].sort_values("drop", ascending=False).to_string(index=False))

    # Bar chart — one bar per perturbation, coloured by coverage
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = merged["has_nppi"].map({True: "#4e79a7", False: "#f28e2b"})
    order  = merged.sort_values("drop", ascending=False)
    bars = ax.bar(range(len(order)), order["drop"], color=colors)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order[pert_col], rotation=45, ha="right", fontsize=9)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_ylabel("pearson_delta drop\n(full − ablated)")
    ax.set_title("Effect of ablating neuronal_PPI per perturbation\n(blue = gene has neuronal_PPI embedding, orange = does not)")

    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#4e79a7", label="Has neuronal_PPI"),
                        Patch(color="#f28e2b", label="No neuronal_PPI")], fontsize=9)
    plt.tight_layout()
    plt.savefig(BASE / "neuronal_ppi_ablation_per_pert.pdf", dpi=150)
    plt.show()

In [ ]:
# ── 4. Compare pearson_delta distributions ───────────────────────────────────
covered_scores   = pert_df[pert_df["has_nppi"]]["pearson_delta"].dropna()
uncovered_scores = pert_df[~pert_df["has_nppi"]]["pearson_delta"].dropna()

stat, pval = stats.mannwhitneyu(covered_scores, uncovered_scores, alternative="two-sided")

print(f"WITH neuronal_PPI    n={len(covered_scores)}  median={covered_scores.median():.4f}  mean={covered_scores.mean():.4f}")
print(f"WITHOUT neuronal_PPI n={len(uncovered_scores)}  median={uncovered_scores.median():.4f}  mean={uncovered_scores.mean():.4f}")
print(f"Mann-Whitney U p={pval:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Boxplot
ax = axes[0]
ax.boxplot([covered_scores, uncovered_scores], labels=["With\nneuronal_PPI", "Without\nneuronal_PPI"],
           patch_artist=True,
           boxprops=dict(facecolor="#76b7b2", alpha=0.7))
ax.set_ylabel("pearson_delta")
ax.set_title(f"Tian: neuronal_PPI coverage effect\n(Mann-Whitney p={pval:.3f})")

# KDE
ax = axes[1]
covered_scores.plot.kde(ax=ax, label=f"With neuronal_PPI (n={len(covered_scores)})", color="#4e79a7")
uncovered_scores.plot.kde(ax=ax, label=f"Without neuronal_PPI (n={len(uncovered_scores)})", color="#f28e2b")
ax.set_xlabel("pearson_delta")
ax.set_title("Distribution comparison")
ax.legend(fontsize=9)
ax.axvline(covered_scores.median(), color="#4e79a7", ls="--", lw=1)
ax.axvline(uncovered_scores.median(), color="#f28e2b", ls="--", lw=1)

plt.tight_layout()
plt.savefig(BASE / "neuronal_ppi_coverage_effect.pdf", dpi=150)
plt.show()

In [ ]:
from scipy import stats

# ── 1. Load neuronal_PPI coverage from the slim npz ──────────────────────────
NPZ = Path("/dcai/users/hilarn/55_cu_0055/data/embeddings/STATE_embedddings/gene_embeddings_slim.npz")
emb = np.load(NPZ, allow_pickle=True)

gene_symbols     = np.array([str(s) for s in emb["gene_symbols"]])   # (G,)
mask_per_source  = emb["mask_per_source"]                              # (G, S)  True=absent
source_names_npz = np.array([str(s) for s in emb["source_names"]])

nppi_idx = np.where(source_names_npz == "neuronal_PPI")[0][0]
has_nppi = ~mask_per_source[:, nppi_idx]                               # (G,) True=has data

covered_symbols = set(gene_symbols[has_nppi])
print(f"neuronal_PPI covers {has_nppi.sum():,} / {len(has_nppi):,} genes")
print(f"Example covered genes: {sorted(covered_symbols)[:10]}")

# ── 2. Load per-perturbation results for Tian full model ─────────────────────
TIAN_EVAL = DATASETS["Tian"] / CKPT_TAG
pert_files = [f for f in TIAN_EVAL.glob("*_results.csv") if "_agg_" not in f.name]

pert_dfs = []
for f in pert_files:
    df = pd.read_csv(f)
    df["cell_type"] = f.name.replace("_results.csv", "")
    pert_dfs.append(df)

pert_df = pd.concat(pert_dfs, ignore_index=True)

# Find perturbation column
pert_col = next((c for c in ["pert", "perturbation", "gene", "condition"]
                 if c in pert_df.columns), None)
if pert_col is None:
    pert_col = [c for c in pert_df.columns
                if not pd.api.types.is_numeric_dtype(pert_df[c]) and c != "cell_type"][0]

print(f"\nPerturbation column: '{pert_col}'")
print(f"Total perturbations: {pert_df[pert_col].nunique()}")

# ── 3. Split into covered vs not covered ────────────────────────────────────
pert_df["has_nppi"] = pert_df[pert_col].isin(covered_symbols)
n_covered   = pert_df[pert_df["has_nppi"]][pert_col].nunique()
n_uncovered = pert_df[~pert_df["has_nppi"]][pert_col].nunique()
print(f"Perturbed genes WITH neuronal_PPI:    {n_covered}")
print(f"Perturbed genes WITHOUT neuronal_PPI: {n_uncovered}")
print(f"\nCovered genes in screen: {sorted(pert_df[pert_df['has_nppi']][pert_col].unique())}")

## neuronal_PPI coverage analysis
Do the perturbed genes that have neuronal_PPI embeddings predict better than matched genes that don't?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# ── Point this at your qc_attn_weights.npz ──────────────────────────────────
NPZ_PATH = Path("/dcai/users/hilarn/55_cu_0055/code/enhance_state/results/30"
                "/qc_emb_30_lr1e-4/qc_emb_30_lr1e-4"
                "/eval_last.ckpt/qc_attn_weights.npz")
# ─────────────────────────────────────────────────────────────────────────────

d = np.load(NPZ_PATH, allow_pickle=True)
weights      = d["attn_weights"]   # (N_cells, N_sources)  float32
pert_names   = d["pert_names"]     # (N_cells,)
celltypes    = d["celltypes"]      # (N_cells,)
source_names = d["source_names"]   # (N_sources,)

print(f"Cells: {weights.shape[0]:,}   Sources: {weights.shape[1]}")
print("Sources:", list(source_names))

## 1 · Overall mean attention per source

In [ ]:
mean_w = weights.mean(axis=0)          # (N_sources,)
order  = np.argsort(mean_w)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(source_names)), mean_w[order], color="steelblue")
ax.set_xticks(range(len(source_names)))
ax.set_xticklabels(source_names[order], rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Mean attention weight")
ax.set_title("Overall source importance (mean across all cells)")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_importance_overall.pdf", dpi=150)
plt.show()

## 2 · Per-source distribution (violin)

In [ ]:
df = pd.DataFrame(weights, columns=source_names)
df_long = df.melt(var_name="source", value_name="attn_weight")

fig, ax = plt.subplots(figsize=(10, 4))
sns.violinplot(data=df_long, x="source", y="attn_weight",
               order=source_names[order], cut=0, ax=ax)
ax.set_xticklabels(source_names[order], rotation=40, ha="right", fontsize=9)
ax.set_ylabel("Attention weight")
ax.set_title("Per-source attention distribution across all cells")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_importance_violin.pdf", dpi=150)
plt.show()

## 3 · Heatmap — mean attention per perturbation

In [ ]:
df["pert"] = pert_names
pert_mean = df.groupby("pert")[list(source_names)].mean()  # (N_perts, N_sources)

# Sort perts by their dominant source
pert_order = pert_mean.values.argmax(axis=1).argsort()
pert_mean_sorted = pert_mean.iloc[pert_order]

fig, ax = plt.subplots(figsize=(max(6, len(source_names) * 0.8),
                                min(40, len(pert_mean) * 0.18 + 2)))
sns.heatmap(pert_mean_sorted[source_names[order]],
            ax=ax, cmap="YlOrRd", linewidths=0,
            xticklabels=True, yticklabels=(len(pert_mean) < 80))
ax.set_title("Mean attention per perturbation × source")
ax.set_xlabel("")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_importance_per_pert.pdf", dpi=150)
plt.show()

print(f"\n{len(pert_mean):,} perturbations")

## 4 · Top-source per perturbation

In [ ]:
top_source = pert_mean[list(source_names)].idxmax(axis=1)
counts = top_source.value_counts()

fig, ax = plt.subplots(figsize=(7, 3))
counts.plot.bar(ax=ax, color="steelblue")
ax.set_ylabel("Number of perturbations")
ax.set_title("Dominant source per perturbation")
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "dominant_source_per_pert.pdf", dpi=150)
plt.show()

print(counts.to_string())

## 5 · Source correlation (do sources co-attend?)

In [ ]:
corr = pd.DataFrame(weights, columns=source_names).corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Cross-source attention correlation")
plt.tight_layout()
plt.savefig(NPZ_PATH.parent / "source_correlation.pdf", dpi=150)
plt.show()